# Interactive renderer review

Replay actual saved Taylor–Green, Sod, Marrone 3.1 and KHH simulations. This notebook does not integrate the solver. If recordings are absent it offers an explicitly labeled analytical example. Run the PyVista section first. Pan/zoom, change fields and modes, hide/show the bar, and save a view for feedback.

Install `pip install -e ".[plot-all]" jupyterlab`. Open from the repository root. Use a fresh kernel when changing Vispy application backends.

Recordings and generated evidence live outside this checkout in the sibling `warpSPHDevelopment/plotting` folder (override `WARPSPH_DEVELOPMENT_ROOT` to relocate it). Run `python examples/regenerate_frontends.py` to recreate the recorded cases. See `reports/README.md` for requirements and limitations.


In [ ]:
from pathlib import Path
import sys, torch, warp as wp
root = Path.cwd()
assert (root / "examples/notebook_review.py").exists(), "Open from the repository root"
if str(root / "examples") not in sys.path:
    sys.path.insert(0, str(root / "examples"))
torch.set_num_threads(2)
wp.init()
from notebook_review import NotebookReview

## PyVista / trame

This uses server rendering so physical colorbar labels and axes match exports. Client rendering currently drops those custom overlays and is not supported by this library. Use the frame slider or Play to replay the recorded motion. The grid uses 96 samples along the longest axis. Normal left dragging pans a 2D view; the mouse wheel zooms.

In [ ]:
import pyvista as pv
from trame_pyvista.jupyter import launch_server
pv.global_theme.trame.server_proxy_enabled = True
pv.global_theme.trame.server_proxy_prefix = "/proxy/"
server = launch_server(host="127.0.0.1")
await server.ready
pv.set_jupyter_backend("trame")
if "review" in globals():
    review.close()
review = NotebookReview(root, backend="pyvista").show()

## Feedback

Try Marrone near the final frame, Sod in Grid view, and Taylor–Green with Streamlines. Check whether camera zoom survives changing the field or palette. Save view + diagnostics writes a PNG, camera state and recent control timings to the external `warpSPHDevelopment/plotting/output/notebook-review/` directory. These timings measure the Python update; they are not browser latency or displayed FPS.

Check responsiveness while playing, drag/zoom accuracy, colorbar label placement, and whether controls or plots become clipped when resizing the browser.

Playback retains your camera. A new case or changed physical domain refits it. Enable **Follow bounds each frame** only when you want continuous automatic fitting. Sod uses a stable sparse viewport by default; its recorded expanding stress-test viewport is used only when following is enabled.

Automatic marker sizes use **Size rule → Physical volume** by default, so rearrangement alone does not shrink the fluid markers. Choose **Close neighbors** to reduce overlap by limiting sizes as close pairs develop. Fixed size remains available by disabling Automatic marker sizes.

## Optional Vispy comparison

Run the next cell to load an interactive `jupyter_rfb` canvas, then run the comparison cell below it. This requires a working OpenGL context (GLFW on this local machine). Do not preselect Vispy EGL/image mode in this kernel.

In [ ]:
import vispy.app
vispy.app.use_app("jupyter_rfb")

In [ ]:
if "review_vispy" in globals():
    review_vispy.close()
review_vispy = NotebookReview(root, backend="vispy").show()

## Cleanup (run when finished)

Close the plots below when done; keep them open while collecting feedback.

In [ ]:
# review.close()
# if "review_vispy" in globals():
#     review_vispy.close()
# await server.stop()